# 🔬 Particle Physics Event Classifier — Training Notebook

**GPU-accelerated training on Google Colab (T4 / A100)**

This notebook:
1. Mounts Google Drive for persistent checkpoints
2. Clones the repo and installs dependencies
3. Downloads the HIGGS dataset (7 GB)
4. Runs ETL → Feature Store → Model Training
5. Logs every run to MLflow (or a local file store)
6. Saves trained models back to Drive

**Runtime:** Select `Runtime → Change runtime type → T4 GPU`

**Expected training times on T4 GPU:**
| Model | Time |
|---|---|
| MLP (500k events, 100 epochs) | ~8–15 min |
| BDT XGBoost | ~5–10 min |
| BDT LightGBM | ~2–5 min |
| Optuna HPO (50 trials) | ~1–2 hrs |
| GNN | ~30–60 min |
| Transformer (ParT) | ~1–3 hrs |


## ⚙️ 0. GPU Check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU detected — switch runtime to T4')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 💾 1. Mount Google Drive (for persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All persistent data lives here — survives runtime restarts
DRIVE_ROOT = '/content/drive/MyDrive/particle_physics_classifier'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/raw',       exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/processed', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/models',         exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/mlruns',         exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/configs',        exist_ok=True)

print(f'Drive root: {DRIVE_ROOT}')
print('Folders created.')

## 📦 2. Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL  = 'https://github.com/rayxxee/Particle-Physics-Classifier.git'
REPO_DIR  = '/content/particle_physics_classifier'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest changes')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install all dependencies (torch is pre-installed on Colab, skip to avoid reinstall)
!pip install -e '.[dev]' --quiet \
    --extra-index-url https://download.pytorch.org/whl/cu118 \
    2>&1 | tail -5

# Verify key imports
import sys
sys.path.insert(0, REPO_DIR)

import torch, numpy, pandas, optuna, mlflow, pandera, omegaconf
print('All imports OK')
print(f'  torch {torch.__version__} | cuda {torch.cuda.is_available()}')
print(f'  optuna {optuna.__version__}')
print(f'  mlflow {mlflow.__version__}')

## 📥 3. Download HIGGS Dataset

The UCI HIGGS dataset is 7 GB compressed. Downloaded once to Drive — subsequent runs skip the download.

In [ ]:
import os

RAW_DIR      = f'{DRIVE_ROOT}/data/raw'
HIGGS_GZ     = f'{RAW_DIR}/HIGGS.csv.gz'
HIGGS_CSV    = f'{RAW_DIR}/HIGGS.csv'
HIGGS_URL    = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'

if os.path.exists(HIGGS_CSV):
    print(f'✅  HIGGS.csv already on Drive ({os.path.getsize(HIGGS_CSV)/1e9:.1f} GB) — skipping download')
elif os.path.exists(HIGGS_GZ):
    print('Decompressing existing .gz ...')
    !gunzip -k {HIGGS_GZ}
    print('Done.')
else:
    print('Downloading HIGGS dataset (~7 GB) — this takes ~5 min on Colab...')
    !wget -q --show-progress -O {HIGGS_GZ} {HIGGS_URL}
    print('Decompressing...')
    !gunzip -k {HIGGS_GZ}
    print('✅  Download complete.')

print(f'CSV size: {os.path.getsize(HIGGS_CSV)/1e9:.2f} GB')

## 🔧 4. Configuration

Edit these settings before running training.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EDIT THESE SETTINGS
# ─────────────────────────────────────────────────────────────────────────────

N_SAMPLES       = 500_000   # None = all 11M events (takes longer)
                             # 100_000 for quick dev, 500_000 for paper-quality

TRAIN_MODEL     = 'mlp'     # 'mlp' | 'bdt' | 'hpo' | 'all'

HPO_N_TRIALS    = 50        # Optuna trials (only used if TRAIN_MODEL='hpo')
HPO_N_EPOCHS    = 30        # Epochs per HPO trial

MLFLOW_TRACKING = True      # Log to MLflow (stored in Drive)

# ─────────────────────────────────────────────────────────────────────────────
# Derived paths (don't edit)
# ─────────────────────────────────────────────────────────────────────────────
PROCESSED_DIR   = f'{DRIVE_ROOT}/data/processed'
FEATURE_DIR     = f'{DRIVE_ROOT}/data/feature_store'
MODELS_DIR      = f'{DRIVE_ROOT}/models'
MLFLOW_DIR      = f'{DRIVE_ROOT}/mlruns'
BEST_CFG_PATH   = f'{DRIVE_ROOT}/configs/mlp_best.yaml'

print('Config:')
print(f'  n_samples : {N_SAMPLES:,}')
print(f'  model     : {TRAIN_MODEL}')
print(f'  mlflow    : {MLFLOW_DIR}')

## 🔄 5. ETL Pipeline

In [ ]:
from src.ingestion.etl_pipeline import ETLConfig, ETLPipeline

etl_config = ETLConfig(
    raw_filepath   = HIGGS_CSV,
    n_samples      = N_SAMPLES,
    processed_dir  = PROCESSED_DIR,
    cache_dir      = f'{DRIVE_ROOT}/data/cache',
    use_reader_cache = True,     # Cache the raw read — big speedup on reruns
)

pipeline = ETLPipeline(etl_config)
splits   = pipeline.run()

print(f'\n✅ ETL complete')
print(f'   Version : {splits.version}')
print(f'   Train   : {splits.n_train:,} events')
print(f'   Val     : {splits.n_val:,} events')
print(f'   Test    : {splits.n_test:,} events')
print(f'   Features: {splits.n_features}')

## 🧮 6. Feature Engineering

In [ ]:
from src.features.feature_store import FeatureConfig, FeatureStore

feat_config = FeatureConfig(
    include_low_level   = True,
    include_dataset_hl  = True,
    include_derived_hl  = True,   # adds ΔR, m_T, H_T, centrality etc.
)

store = FeatureStore(feat_config, cache_dir=FEATURE_DIR)
store.build(splits)

X_train, y_train = store.load('train')
X_val,   y_val   = store.load('val')
X_test,  y_test  = store.load('test')

print(f'\n✅ Features ready')
print(f'   Train shape : {X_train.shape}')
print(f'   Feature cols: {list(X_train.columns[:5])} ...')

## 📊 7. MLflow Setup

In [ ]:
import mlflow

if MLFLOW_TRACKING:
    mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
    mlflow.set_experiment('particle_physics_classifier')
    print(f'MLflow tracking URI: file://{MLFLOW_DIR}')
    print('View results after training: mlflow ui --backend-store-uri', MLFLOW_DIR)
else:
    print('MLflow tracking disabled')

## 🧠 8a. Train MLP Baseline

In [ ]:
if TRAIN_MODEL in ('mlp', 'all'):
    from src.models.mlp.config import MLPConfig
    from src.models.mlp.model  import MLPModel
    from sklearn.metrics import roc_auc_score
    import numpy as np

    mlp_config = MLPConfig(
        input_dim      = X_train.shape[1],
        hidden_dims    = [512, 256, 128, 64],
        dropout_rates  = [0.3, 0.3, 0.2, 0.0],
        batch_norm     = True,
        epochs         = 100,
        batch_size     = 4096,
        learning_rate  = 1e-3,
        weight_decay   = 1e-4,
        scheduler_name = 'cosine_annealing',
        early_stopping = True,
        early_stopping_patience = 10,
        mixed_precision = True,   # ← AMP on GPU gives 30–50% speedup
        seed = 42,
    )

    mlp = MLPModel(mlp_config)

    with mlflow.start_run(run_name='mlp_baseline') if MLFLOW_TRACKING else __import__('contextlib').nullcontext():
        if MLFLOW_TRACKING:
            mlflow.log_params(mlp_config.to_dict())

        best_auc = mlp.fit(
            X_train.values, y_train.values,
            X_val.values,   y_val.values,
        )

        # Test evaluation
        scores   = mlp.predict_proba(X_test.values)
        test_auc = roc_auc_score(y_test.values, scores)

        print(f'\n✅ MLP Training complete')
        print(f'   Best val AUC : {best_auc:.4f}')
        print(f'   Test AUC     : {test_auc:.4f}')

        if MLFLOW_TRACKING:
            mlflow.log_metric('best_val_auc', best_auc)
            mlflow.log_metric('test_auc', test_auc)

    # Save to Drive
    mlp_save_path = f'{MODELS_DIR}/mlp_baseline'
    mlp.save(mlp_save_path)
    print(f'   Saved to     : {mlp_save_path}')

## 🌲 8b. Train BDT Baselines (XGBoost + LightGBM)

In [ ]:
if TRAIN_MODEL in ('bdt', 'all'):
    try:
        import xgboost as xgb
        import lightgbm as lgb
        from sklearn.metrics import roc_auc_score
        import numpy as np
        import joblib, os

        X_tr = X_train.values.astype('float32')
        y_tr = y_train.values.astype('float32')
        X_v  = X_val.values.astype('float32')
        y_v  = y_val.values.astype('float32')
        X_te = X_test.values.astype('float32')
        y_te = y_test.values.astype('float32')

        # ── XGBoost ──────────────────────────────────────────────────────────
        print('Training XGBoost...')
        xgb_model = xgb.XGBClassifier(
            n_estimators     = 1000,
            max_depth        = 6,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            use_label_encoder = False,
            eval_metric      = 'auc',
            early_stopping_rounds = 20,
            tree_method      = 'gpu_hist' if __import__('torch').cuda.is_available() else 'hist',
            random_state     = 42,
            n_jobs           = -1,
        )
        xgb_model.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=50)
        xgb_auc = roc_auc_score(y_te, xgb_model.predict_proba(X_te)[:, 1])
        print(f'✅  XGBoost test AUC: {xgb_auc:.4f}')

        # ── LightGBM ──────────────────────────────────────────────────────────
        print('\nTraining LightGBM...')
        lgb_model = lgb.LGBMClassifier(
            n_estimators     = 1000,
            max_depth        = 6,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            num_leaves       = 63,
            random_state     = 42,
            n_jobs           = -1,
            device           = 'gpu' if __import__('torch').cuda.is_available() else 'cpu',
        )
        lgb_model.fit(
            X_tr, y_tr,
            eval_set = [(X_v, y_v)],
            callbacks = [lgb.early_stopping(20), lgb.log_evaluation(50)],
        )
        lgb_auc = roc_auc_score(y_te, lgb_model.predict_proba(X_te)[:, 1])
        print(f'✅  LightGBM test AUC: {lgb_auc:.4f}')

        # Save
        joblib.dump(xgb_model, f'{MODELS_DIR}/xgboost.pkl')
        joblib.dump(lgb_model, f'{MODELS_DIR}/lightgbm.pkl')

        if MLFLOW_TRACKING:
            with mlflow.start_run(run_name='bdt_baselines'):
                mlflow.log_metric('xgb_test_auc', xgb_auc)
                mlflow.log_metric('lgb_test_auc', lgb_auc)

    except ImportError:
        print('xgboost/lightgbm not installed. Run: pip install xgboost lightgbm')

## 🔍 8c. Hyperparameter Optimization (Optuna)

In [ ]:
if TRAIN_MODEL == 'hpo':
    from src.models.mlp.optimizer import HPOConfig, MLPOptimizer
    import numpy as np

    hpo_cfg = HPOConfig(
        n_trials           = HPO_N_TRIALS,
        n_epochs_per_trial = HPO_N_EPOCHS,
        study_name         = 'mlp_hpo_colab',
        storage            = f'sqlite:///{DRIVE_ROOT}/optuna.db',   # Persisted to Drive!
        save_best_config   = BEST_CFG_PATH,
        n_startup_trials   = 10,
        n_warmup_steps     = 5,
        seed               = 42,
    )

    optimizer = MLPOptimizer(hpo_cfg)
    best_cfg  = optimizer.run(
        X_train.values, y_train.values,
        X_val.values,   y_val.values,
    )

    print('\n✅  HPO complete')
    print(f'   Best val AUC : {optimizer.study.best_value:.4f}')
    print(f'   Best params  : {optimizer.study.best_params}')
    print(f'   Config saved : {BEST_CFG_PATH}')

    # Retrain final model with best config (full epochs)
    from src.models.mlp.model import MLPModel
    from sklearn.metrics import roc_auc_score

    print('\nRetraining with best config (full epochs)...')
    best_cfg.epochs = 100
    best_cfg.mixed_precision = True
    final_model = MLPModel(best_cfg)
    final_model.fit(
        X_train.values, y_train.values,
        X_val.values,   y_val.values,
    )
    test_auc = roc_auc_score(y_test.values, final_model.predict_proba(X_test.values))
    print(f'✅  Final test AUC: {test_auc:.4f}')

    final_model.save(f'{MODELS_DIR}/mlp_hpo_best')

    # Importance plot
    import matplotlib.pyplot as plt
    imp = optimizer.importance()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(list(imp.keys()), list(imp.values()), color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title('Hyperparameter Importance (Optuna)')
    plt.tight_layout()
    plt.savefig(f'{DRIVE_ROOT}/hpo_importance.png', dpi=150)
    plt.show()
    print('Importance plot saved to Drive')

## 📈 9. Evaluation & Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve

results = {}  # model_name → test scores

# Collect scores for all trained models
try:
    results['MLP'] = mlp.predict_proba(X_test.values)
except NameError:
    pass

try:
    results['XGBoost']  = xgb_model.predict_proba(X_test.values)[:, 1]
    results['LightGBM'] = lgb_model.predict_proba(X_test.values)[:, 1]
except NameError:
    pass

y_true = y_test.values

if results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

    # ── ROC curves
    ax = axes[0]
    for (name, scores), color in zip(results.items(), colors):
        fpr, tpr, _ = roc_curve(y_true, scores)
        auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.4f})', color=color, lw=2)
    ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

    # ── Score distributions
    ax = axes[1]
    name = list(results.keys())[0]
    scores = results[name]
    ax.hist(scores[y_true==0], bins=50, alpha=0.6, label='Background', color='#F44336', density=True)
    ax.hist(scores[y_true==1], bins=50, alpha=0.6, label='Signal',     color='#2196F3', density=True)
    ax.set_xlabel('Classifier Score')
    ax.set_ylabel('Density')
    ax.set_title(f'{name} — Score Distribution')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.suptitle('Particle Physics Event Classifier — Results', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_ROOT}/results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved to {DRIVE_ROOT}/results.png')
else:
    print('No trained models found — run training cells above first')

## 💾 10. Save Summary to Drive

In [ ]:
from sklearn.metrics import roc_auc_score
import json, datetime

summary = {
    'timestamp'   : datetime.datetime.now().isoformat(),
    'n_samples'   : N_SAMPLES,
    'n_features'  : X_train.shape[1],
    'models'      : {},
}

for name, scores in results.items():
    auc_val = roc_auc_score(y_true, scores)
    summary['models'][name] = {'test_auc': round(auc_val, 6)}
    print(f'  {name:12s}  AUC = {auc_val:.4f}')

with open(f'{DRIVE_ROOT}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n✅  Summary saved to {DRIVE_ROOT}/training_summary.json')

## 📌 Notes

- **Models** are saved to `MyDrive/particle_physics_classifier/models/`
- **Plots** are at `MyDrive/particle_physics_classifier/results.png`
- **Optuna study** is at `MyDrive/particle_physics_classifier/optuna.db` — resume HPO any time
- **MLflow runs** are at `MyDrive/particle_physics_classifier/mlruns/`
- Re-run from **Cell 5 (ETL)** on restart — data is cached on Drive
- For GNN/Transformer, upgrade to **Colab Pro** for A100 access
